# Repository edition: 10_signed_projection

Run from this repository in your existing GPU environment. Source is in `../suites/10_signed_projection/`. The launcher creates a versioned working copy. No embedded base64 code or installation is needed. Review configuration before launching; optional stages are not all enabled by default. See `../docs/EXPERIMENTS.md` for paper mapping.


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
SUPPLEMENTARY = next((p for p in _candidates if (p / "tools" / "notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    SUPPLEMENTARY = next((p / "supplementary" for p in _candidates if (p / "supplementary/tools/notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    raise FileNotFoundError("Open this notebook from inside the extracted supplementary repository.")
sys.path.insert(0, str(SUPPLEMENTARY / "tools"))
from notebook_sources import source_bundle as _source_bundle


# Signed residual-mean projections under singleton freezes
Full code and matched cohort/model pins embedded. **No install or HF token setup.**

Runs five conditions on the same prompts: **live, attention frozen, gates frozen, normalization scales frozen, all frozen**. Measures the actual signed projection of each residual gradient onto an independent mean-residual direction, plus paired removal effects. Defaults:100 prompts each on trained Llama1B and8B, matching the previous anisotropy selection.

Primary effect: `live projection − singleton-freeze projection`. Normalization-minus-gate and normalization-minus-attention contrasts are paired within prompt. These effects overlap; they are not additive shares or demonstrations of factual repair.


In [ ]:
from pathlib import Path
import base64,hashlib,io,json,os,subprocess,sys,time,uuid,zipfile
BUNDLE_SHA256 = hashlib.sha256(base64.b64decode(_source_bundle('10_signed_projection'))).hexdigest()
BUNDLE = _source_bundle('10_signed_projection')

payload=base64.b64decode(BUNDLE)
assert hashlib.sha256(payload).hexdigest()==BUNDLE_SHA256
CODE=Path.home()/('threeways_projection_code_'+BUNDLE_SHA256[:12]);CODE.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for item in z.infolist():
        if CODE.resolve() not in (CODE/item.filename).resolve().parents:raise ValueError('Unsafe archive path')
    z.extractall(CODE)
EVIDENCE=CODE/'evidence'
print('Ready:',CODE)


## 1. Existing GPU Python

In [ ]:
candidates=[Path('/home/ubuntu/venvs/threeways/bin/python'),Path(sys.executable),Path.home()/'threeways_causal_env/bin/python']
check="import torch,transformers,numpy,pandas,matplotlib,huggingface_hub; assert transformers.__version__=='4.48.3', 'Expected tested Transformers4.48.3'; assert torch.cuda.is_available(), 'CUDA unavailable'; print(torch.cuda.get_device_name())"
PYTHON=None;errors=[]
for candidate in dict.fromkeys(candidates):
    if not candidate.exists():continue
    p=subprocess.run([str(candidate),'-c',check],capture_output=True,text=True)
    if p.returncode==0:PYTHON=candidate;print('Using:',candidate,'|',p.stdout.strip());break
    errors.append(str(candidate)+': '+p.stderr[-1000:])
if PYTHON is None:raise RuntimeError('Point candidates to your working Python.\n'+'\n'.join(errors))


## 2. Configure ONCE
Default100 evaluation prompts,100 separate reference subjects,1B and8B. To run only1B, change `CONFIG['models']` before launching. This is the targeted experiment, not a rerun of the previous campaign.

Five backward passes per evaluation prompt. Small finite-difference checks cover two prompts per model, three depths, three step sizes, all five conditions. No wall-time estimate has been benchmarked.

Compact gradient vectors are retained for future analysis (roughly313MiB before compression across both models). They are excluded from the small export.


In [ ]:
from datetime import datetime
import shutil
if 'RESULTS' in globals():raise RuntimeError(f'Campaign already selected: {RESULTS}. Use monitor/export. Do not rerun configuration.')
CONFIG=json.loads((CODE/'projection_study/config.json').read_text())
# Optional changes BEFORE first launch:
# CONFIG['models']=['llama1b']
# CONFIG['prompts']=None  # Full2,995 prompts, substantially more work.
# CONFIG['save_gradients']=False
RUN_NAME='signed_projection_'+datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]
RESULTS=Path.home()/'threeways_projection_results'/RUN_NAME
RESULTS.mkdir(parents=True,exist_ok=False)
CONFIG_PATH=RESULTS/'configuration.json';CONFIG_PATH.write_text(json.dumps(CONFIG,indent=2))
(RESULTS/'bundle.json').write_text(json.dumps({'bundle_sha256':BUNDLE_SHA256}))
print('Selected results:',RESULTS)
print('Models:',CONFIG['models'],'| prompts/model:',CONFIG['prompts'])
print('Free disk GiB:',round(shutil.disk_usage(RESULTS).free/2**30,1))


## 3. Launch detached worker

In [ ]:
import fcntl
if (RESULTS/'launch.json').exists():raise RuntimeError('Already launched. Use monitor or explicit resume below.')
if shutil.disk_usage(RESULTS).free<20*2**30:raise RuntimeError('Less than20GiB free; free space or use another output disk first.')
command=[str(PYTHON),'-u','-m','projection_study.supervisor','--config',str(CONFIG_PATH),'--out',str(RESULTS),'--evidence',str(EVIDENCE)]
with (RESULTS/'supervisor.log').open('a') as log:
    supervisor=subprocess.Popen(command,cwd=CODE,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
print('Supervisor PID:',supervisor.pid)
print('Keep the GPU instance powered on. Closing the browser is fine.')


## 4. Monitor

In [ ]:
def tail(path,n=7000):
    if path.exists():
        with path.open('rb') as f:
            f.seek(max(0,path.stat().st_size-n));print(f.read().decode(errors='replace'))
tail(RESULTS/'execution.log')
for model in CONFIG['models']:
    count=sum(1 for _ in (RESULTS/model/'prompts').glob('*/measurement.json'))
    print(model,':',count,'completed prompts')
p=RESULTS/'exit.json'
print('WORKER EXIT:',json.loads(p.read_text()) if p.exists() else 'Not recorded yet')
if not (RESULTS/'execution.log').exists():tail(RESULTS/'supervisor.log')


## 5. Inspect checks and paired contrasts
Positive normalization-minus-gate means permitting normalization contributed a more positive mean-direction component in this paired comparison. This is not an additive causal allocation. Inspect raw projections alongside the effects.


In [ ]:
import csv
p=RESULTS/'reports/audit.json'
if p.exists():print('Audit:',json.loads(p.read_text()))
p=RESULTS/'reports/paired_contrasts_summary.csv'
if p.exists():
    with p.open() as f:
        for i,row in enumerate(csv.DictReader(f)):
            if i>=12:break
            print({k:row[k] for k in ['model','layer','norm_minus_gate','norm_minus_gate_lo','norm_minus_gate_hi',
                                      'norm_minus_attention','norm_minus_attention_lo','norm_minus_attention_hi']})
print('Figures and all tables:',RESULTS/'reports')


## 6. Optional stop / explicit resume
Stop is cooperative; let the current prompt/check batch finish. To resume, wait for worker exit, then use the commented resume code. Same settings and code are required. Completed records are retained.


In [ ]:
# Request stop:
# (RESULTS/'STOP').touch()

# Explicit resume only after the worker has exited:
# if not (RESULTS/'exit.json').exists():raise RuntimeError('Wait for worker exit first.')
# for name in ['run.lock','supervisor.lock']:
#     with (RESULTS/name).open('a+') as handle:
#         fcntl.flock(handle,fcntl.LOCK_EX|fcntl.LOCK_NB)
# (RESULTS/'STOP').unlink(missing_ok=True)
# with (RESULTS/'supervisor.log').open('a') as log:
#     supervisor=subprocess.Popen([str(PYTHON),'-u','-m','projection_study.supervisor','--config',str(CONFIG_PATH),
#         '--out',str(RESULTS),'--evidence',str(EVIDENCE),'--resume'],cwd=CODE,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)


## 7. Small complete export
Successful execution does not guarantee every finite-difference check passed. Export includes checks, per-prompt scalar measurements, paired summaries, figures, reference means and provenance. It excludes gradient caches and model weights. Download before terminating the GPU instance.


In [ ]:
from IPython.display import display,FileLink
ROOT=RESULTS
p=ROOT/'exit.json';status=json.loads(p.read_text()) if p.exists() else None
if not status or status.get('returncode')!=0 or status.get('stopped') or not (ROOT/'completion.json').exists():
    raise RuntimeError('Worker is not fully complete. Use monitor first.')
files=set()
for pattern in ['reports/*.csv','reports/*.json','reports/*.pdf','reports/*.png','*/model.json',
                '*/reference.json','*/reference_mean.npz','data/cohort.json']:
    files.update(p for p in ROOT.glob(pattern) if p.is_file())
for name in ['configuration.json','protocol.json','environment.json','completion.json','exit.json','bundle.json']:
    if (ROOT/name).exists():files.add(ROOT/name)
if not any(p.suffix=='.csv' for p in files):raise RuntimeError('No tables found. Export cancelled.')
archive=Path.cwd()/(ROOT.name+'_COMPLETE_summaries_'+datetime.now().strftime('%Y%m%d_%H%M%S')+'.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(files):z.write(p,p.relative_to(ROOT))
    z.writestr('EXPORT_MANIFEST.json',json.dumps({'source':str(ROOT),'files':len(files),'bundle_sha256':BUNDLE_SHA256}))
    for name in ['execution.log','supervisor.log']:
        p=ROOT/name
        if p.exists():
            with p.open('rb') as f:
                f.seek(max(0,p.stat().st_size-20000));z.writestr('log_tails/'+name,f.read())
with zipfile.ZipFile(archive) as z:assert z.testzip() is None
print('Verified:',len(files),'files |',round(archive.stat().st_size/2**20,2),'MiB')
display(FileLink(archive.name))


## Optional numerical tests (tiny CPU models; no downloads)

In [ ]:
# subprocess.run([str(PYTHON),'-m','unittest','discover','-s','tests','-v'],cwd=CODE,check=True)
